<a href="https://colab.research.google.com/github/fyremael/TROPICA/blob/main/notebooks/01_operator_onboarding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TROPICA Operator Onboarding

Goal: install TROPICA, run the evidence pipeline, inspect the generated dashboards, and leave with a concrete sense of what the pass/fail gates prove.

TROPICA treats constrained decoding as an evidence-backed control plane. The model or logit source may propose anything, but the external support contract decides which token IDs are legal. This notebook gives operators the shortest path from a blank runtime to a passing evidence report.

In [ ]:
import pathlib
import subprocess
import sys

repo_root = pathlib.Path.cwd()
if (repo_root / "pyproject.toml").exists():
    install_cmd = [sys.executable, "-m", "pip", "install", "-q", "-e", ".[real-tokenizers,dev]"]
else:
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "control-delta-support-decoding[real-tokenizers,dev] @ git+https://github.com/fyremael/TROPICA.git",
    ]

print("Installing:", " ".join(install_cmd))
subprocess.check_call(install_cmd)


In [ ]:
import cdsd

expected_exports = [
    "ToolCallSpec",
    "StructuredOutputCompiler",
    "StructuredOutputDecoder",
    "TiktokenAdapter",
    "HostileLogitProvider",
]
missing = [name for name in expected_exports if not hasattr(cdsd, name)]
print("TROPICA exports available:", expected_exports)
assert not missing, missing


Run the full report command. In a source checkout this can include tests; in a plain installed Colab runtime the evidence harnesses still run and validate the generated artifacts.

In [ ]:
import json
import pathlib
import subprocess

artifact_dir = pathlib.Path("colab_operator_artifacts")
cmd = ["cdsd-report", "--artifacts", str(artifact_dir), "--jobs", "4"]
if pathlib.Path("tests").exists():
    cmd.append("--with-pytest")
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)

manifest = json.loads((artifact_dir / "report_manifest.json").read_text())
failed = [gate for gate in manifest["gates"] if not gate["passed"]]
print("passed:", manifest["passed"])
print("commands:", len(manifest["commands"]))
print("gates:", len(manifest["gates"]))
print("failed gates:", failed)
assert manifest["passed"]


In [ ]:
from IPython.display import Markdown, SVG, display

display(Markdown((artifact_dir / "report_index.md").read_text()))
display(SVG(filename=str(artifact_dir / "model_integration_visuals.svg")))


Interpretation: a passing run means the install can execute the evidence suite, produce CSV/Markdown/SVG artifacts, and satisfy the hard gates for experiments, tokenizer correctness, structured output, offline model integration, stress, and scale. The dashboard is not decorative; it is the visual front door to the same gated data in `report_manifest.json`.